# Lab03 — Frozen DistilBERT: thử giảm overfitting, đủ 10 epoch

Chạy lần lượt 10 cell code trong session mới, GPU và Internet bật. Train từ pretrained base, không resume checkpoint cũ. Chưa có kết quả GPU cho cấu hình này; không bảo đảm hết overfitting.

| Phase | Công việc | Module/file tham chiếu | Artifact |
|---|---|---|---|
| 1 | Repo và Python 3.12 | requirements.txt | Kiểm tra GPU |
| 2 | Chốt cấu hình trước test | config/frozen_trial.json | Cấu hình |
| 3 | Tests + sanity fixture | tests/test_frozen.py, test_training.py | outputs/preflight |
| 4 | Pretrained sentiment | scripts/run_exercise1.py | Exercise1 PNG |
| 5 | Full dataset EDA | scripts/preview_dataset.py | Dataset PNG |
| 6 | Augmentation có lọc | scripts/preview_augmentation.py | Counts/examples PNG |
| 7 | Train 10 epoch | scripts/run_frozen_trial.py, src/training/frozen.py | Epoch history/learning curves |
| 8 | Checkpoint và threshold | validation loss, validation Macro-F1 | Selection/threshold PNG |
| 9 | Test sau khóa policy | TF-IDF và frozen DistilBERT | Confusion/ROC/PR/calibration PNG |
| 10 | Audit và ZIP | manifest SHA256 | Review ZIP + checkpoint ZIP riêng |

Giữ augmentation hiện tại; không ép thêm 800 câu. Đây là thay đổi nhiều yếu tố so với lịch sử, chưa phải ablation chứng minh tác dụng riêng từng yếu tố. Test đã được xem ở các thí nghiệm trước nên chỉ báo cáo mô tả; lựa chọn mới chỉ dựa validation.


## 1. Clone repository, môi trường riêng và GPU

Python kernel của Kaggle có thể khác Python của dependencies. Dùng venv Python 3.12 riêng và gọi child interpreter cho mọi xử lý ML; ảnh được hiển thị trong kernel cha. Repo tồn tại thì giữ nguyên kết quả, không reset/xóa dữ liệu. Nếu code cũ, cập nhật branch lab03 trước khi chạy. Chỉ GPU đầu tiên được dùng để giữ batch hiệu dụng ổn định.

In [ ]:
import os
import sys
import json
import shutil
import subprocess
import importlib.util
from pathlib import Path
from IPython.display import display, Image, Markdown, FileLink

REPO_URL = "https://github.com/lacthui06/Lab-DL.git"
BRANCH = "lab03"
REPO = Path("/kaggle/working/Lab-DL")
LAB = REPO / "labs/lab03"
ENV = Path("/kaggle/working/lab03-py312")
PYTHON = str(ENV / "bin/python")

assert shutil.which("nvidia-smi"), "Hãy bật Settings > Accelerator > GPU."
subprocess.run(["nvidia-smi"], check=True)
if not REPO.exists():
    subprocess.run(["git", "clone", "--single-branch", "--branch", BRANCH, REPO_URL, str(REPO)], check=True)
else:
    print("Giữ repo và các kết quả hiện có.")
assert (LAB / "scripts/preview_dataset.py").is_file(), "Hãy cập nhật code lab03 phiên bản 10 cell."
subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, check=True)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv==0.9.5"], check=True)
def uv(*args):
    subprocess.run([sys.executable, "-m", "uv", *map(str,args)], check=True)
uv("python", "install", "3.12")
if not Path(PYTHON).is_file(): uv("venv", "--python", "3.12", ENV)
uv("pip", "install", "--python", PYTHON, "--link-mode", "copy", "-r", LAB / "requirements.txt")
uv("pip", "check", "--python", PYTHON)

RUN_ENV = os.environ.copy()
RUN_ENV.update(CUDA_VISIBLE_DEVICES="0", TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1",
               CUBLAS_WORKSPACE_CONFIG=":4096:8", MPLCONFIGDIR="/kaggle/working/.mpl-lab03")
def run_python(*args, env=None):
    return subprocess.run([PYTHON,"-u",*map(str,args)], cwd=REPO,
                          env=RUN_ENV if env is None else env, check=True)
def selected_folder(kind):
    state=json.loads((LAB / "config/workflow.json").read_text())
    folder=(LAB / "outputs/runs" / state["active_runs"][kind]).resolve()
    assert folder.parent==(LAB / "outputs/runs").resolve(), "Run path không hợp lệ."
    return folder

def show_figures(folder, names=None):
    folder=Path(folder)
    paths=[folder/name for name in names] if names else sorted(folder.glob("*.png"))
    assert paths, f"Không có PNG trong {folder}"
    for path in paths:
        assert path.is_file(), f"Thiếu ảnh {path}"
        display(Markdown(f"**{path.name}**"))
        display(Image(filename=str(path)))

run_python("-c", "import sys,torch,transformers; print(sys.version); print(torch.__version__,transformers.__version__); assert torch.cuda.is_available(), 'GPU chưa khả dụng'; print(torch.cuda.get_device_name(0))")
free_gib=shutil.disk_usage("/kaggle/working").free/1024**3
print(f"Dung lượng trống: {free_gib:.1f} GiB")
assert free_gib>=6, "Cần ít nhất 6 GiB trống; hãy dùng session sạch."

## 2. Cấu hình đã định trước

Đóng băng embedding và 4/6 tầng encoder đầu. Chỉ cập nhật hai tầng cuối cùng và classification head. Encoder LR 2e-6, head LR 1e-5; warmup 10%, linear decay. Dropout 0.2/0.2/0.4, weight decay 0.1 (bias/LayerNorm không decay). Chạy đủ 10 epoch, không early stopping. Checkpoint có validation loss thấp nhất được chọn trước khi quét threshold validation Macro-F1. Mục tiêu giảm overfitting, đồng thời tránh underfitting.


In [ ]:
import pandas as pd
CONFIG_FILE = LAB / "config/frozen_trial.json"
assert CONFIG_FILE.is_file(), "Repo chưa có frozen_trial.json; hãy cập nhật code rồi import notebook mới."
config = json.loads(CONFIG_FILE.read_text())
assert config["exercise2"]["num_train_epochs"] == 10
assert config["exercise2"]["metric_for_best_model"] == "loss"
display(pd.DataFrame([{k:v for k,v in config["exercise2"].items() if not isinstance(v,dict)}]))
display(config["exercise2"]["model_overrides"])
display(config["exercise2"]["augmentation"])


## 3. Preflight tests và sanity overfit

Tests dùng fixture cực nhỏ trên CPU để kiểm tra shape, loss/gradient, 10 epoch, augmentation và artifact. Không fine-tune full dataset trên CPU và không coi fixture là benchmark. Bước này phải pass trước training; XML và toàn bộ log được lưu trong outputs/preflight/.

In [ ]:
from datetime import datetime, timezone
validation_dir=LAB / "outputs/preflight" / datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
validation_dir.mkdir(parents=True,exist_ok=False)
TEST_ENV=RUN_ENV.copy(); TEST_ENV["CUDA_VISIBLE_DEVICES"]=""
with (validation_dir/"pytest.log").open("w",encoding="utf-8") as log:
    process=subprocess.Popen([PYTHON,"-u","-m","pytest","labs/lab03/tests","-q",
        f"--junitxml={validation_dir/'pytest.xml'}"],cwd=REPO,env=TEST_ENV,
        stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True)
    for line in process.stdout:
        print(line,end="");log.write(line)
    assert process.wait()==0, "Preflight failed; xem pytest.log, chưa chạy training."
print("Preflight PASSED:",validation_dir)

## 4. Exercise 1: tokenization và pretrained sentiment

Đối chiếu pipeline với manual forward/softmax trên cùng model và câu đầu vào. In token IDs, attention mask và probabilities; xem ảnh token heatmap và bảng so sánh. SST-2 pretrained phục vụ Exercise 1; Exercise 2 tải DistilBERT base và khởi tạo classification head mới.

In [ ]:
run_python("labs/lab03/scripts/run_exercise1.py")
EXERCISE1=selected_folder("exercise1")
EXERCISE1_ANALYSIS=selected_folder("exercise1_analysis")
display(pd.DataFrame(json.loads((EXERCISE1/"predictions.json").read_text())["manual"]))
display(json.loads((EXERCISE1/"tokenizer_breakdown.json").read_text()))
show_figures(EXERCISE1_ANALYSIS)

## 5. Full dataset và EDA trước augmentation

Xác nhận train/validation/test 8.530/1.066/1.066, nhãn nhị phân và phân bố lớp. Độ dài token chỉ đo train để giải thích max_length=128; mẫu minh họa cũng chỉ lấy train. Không fit tokenizer/vocabulary trên heldout. EDA được lưu thành run dataset_preview và đi cùng review ZIP nếu cùng dataset/tokenizer config.

In [ ]:
run_python("labs/lab03/scripts/preview_dataset.py","--config",CONFIG_FILE)
EDA=selected_folder("dataset_preview")
eda=json.loads((EDA/"data_preview.json").read_text())
display(pd.DataFrame([{"split":k,"samples":v,"negative":eda["label_counts"][k]["0"],
                      "positive":eda["label_counts"][k]["1"]} for k,v in eda["split_sizes"].items()]))
display(pd.DataFrame([eda["train_token_lengths"]]))
show_figures(EDA)

## 6. Augmentation: số mẫu và ví dụ trước/sau

Giữ mọi mẫu train gốc, thêm tối đa 25% synthetic variants. Thay một từ/cụm từ cùng cực tính, bỏ câu phủ định/tương phản, chặn trùng và cân bằng số mẫu thêm. Nhãn kế thừa có thể có noise: đọc bảng ví dụ trước khi tiếp tục. Không gọi synthetic variants là ground truth mới đã được gán nhãn thủ công.

Validation/test không tăng mẫu. Số thêm thực tế không nhất thiết đạt mức tối đa 2.132; không nhân bản nguyên xi để đủ quota. Baseline và Transformer dùng cùng augmented pool. Số step/epoch được tính theo effective train size, trong khi hình chính vẫn có 10 điểm epoch.

In [ ]:
run_python("labs/lab03/scripts/preview_augmentation.py","--config",CONFIG_FILE)
PREVIEW=selected_folder("augmentation_preview")
augmentation=json.loads((PREVIEW/"augmentation.json").read_text())
display(pd.DataFrame([{"official_train":augmentation["original_train_size"],
    "synthetic_added":augmentation["added_count"], "effective_train":augmentation["effective_train_size"],
    "validation":eda["split_sizes"]["validation"], "test":eda["split_sizes"]["test"]}]))
display(pd.DataFrame(augmentation["records"]).head(8))
show_figures(PREVIEW)

## 7. Train mới từ pretrained base, đủ 10 epoch

Một cấu hình được định trước; không chạy search trên test. TF-IDF và Transformer dùng cùng augmented train pool. Mỗi epoch đo full train/validation trong eval mode (dropout tắt khi đo), ghi đúng 10 dòng và 10 điểm. LR hiển thị trong bảng epoch là nhóm encoder; head dùng lịch cùng tỷ lệ nhưng LR gốc khác. Cell này thực hiện cả train, khóa policy rồi test; cell 8–9 chỉ trình bày artifact đã ghi. Không chạy lại cell 7 nếu chỉ muốn xem ảnh.


In [ ]:
run_python("labs/lab03/scripts/run_frozen_trial.py", "--config", CONFIG_FILE)
TRIAL = selected_folder("frozen_trial")
display(pd.DataFrame(json.loads((TRIAL/"epoch_history.json").read_text())))
show_figures(TRIAL, ["learning_curves.png"])


## 8. Đọc quyết định checkpoint và threshold đã cố định

Checkpoint chọn theo validation loss, không phải epoch 10 mặc định. Threshold chọn theo validation Macro-F1 sau khi cố định checkpoint. Nếu khoảng cách train–validation vẫn lớn, cần báo cáo đúng; không sửa số liệu để đường học đẹp hơn.


In [ ]:
display(json.loads((TRIAL/"selection_summary.json").read_text()))
display(json.loads((TRIAL/"freezing_and_optimizer.json").read_text()))
display(json.loads((TRIAL/"frozen_selection.json").read_text()))
show_figures(TRIAL, ["validation_threshold_curves.png"])


## 9. Test và chẩn đoán xác suất

Xem Macro-F1 cùng validation loss và calibration; không chỉ xem train–validation gap. Baseline được fit train-only, chọn C/threshold bằng validation. Cấu hình đã khóa trước test. Một seed chưa đo độ biến thiên giữa các lượt train.


In [ ]:
display(pd.DataFrame(json.loads((TRIAL/"test_comparison.json").read_text())))
display(json.loads((TRIAL/"test_runtime_metrics.json").read_text()))
display(json.loads((TRIAL/"probability_diagnostics.json").read_text()))
show_figures(TRIAL, ["confusion_matrix.png", "roc_pr_curves.png", "calibration_confidence.png"])
predictions = pd.DataFrame(json.loads((TRIAL/"test_predictions.json").read_text()))
errors = pd.DataFrame(json.loads((TRIAL/"threshold_errors.json").read_text()))
if not errors.empty:
    display(errors.merge(predictions, on=["index", "true_label"]).head(12))


## 10. Kiểm tra ZIP và tải kết quả

Review ZIP chứa config, số liệu, dự đoán, ảnh, log, source hashes và manifest. Checkpoint ZIP chứa best_model để dùng inference hoặc đánh giá lại mà không train lại; giữ riêng, không commit trọng số nặng vào Git. Script đóng gói không đọc notebook đang mở: tải thêm File > Download Notebook để giữ output hiển thị.


In [ ]:
import hashlib
import zipfile
review = Path("/kaggle/working") / f"lab03_review_{TRIAL.name}.zip"
checkpoint = Path("/kaggle/working") / f"lab03_checkpoint_{TRIAL.name}.zip"
assert review.is_file() and checkpoint.is_file()
with zipfile.ZipFile(review) as archive:
    manifest = json.loads(archive.read("bundle_manifest.json"))
    assert all(hashlib.sha256(archive.read(name)).hexdigest() == digest for name,digest in manifest.items())
print("SHA256 review manifest: PASSED")
for path in (review, checkpoint):
    print(path, f"{path.stat().st_size / 1024**2:.1f} MiB")
    display(FileLink(str(path)))
print("Gửi review ZIP để kiểm tra; giữ checkpoint ZIP riêng. Tải thêm notebook có output qua File > Download Notebook.")
